In [13]:
# ============ PIVOT TRANSLATION — TEK HÜCREDE TÜM PIPELINE ============
import re
import torch
import numpy as np
from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, MarianMTModel, MarianTokenizer
from datasets import load_dataset, Dataset
from tqdm import tqdm
import torch.nn.functional as F

# 1. Cihaz + ayarlar
device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
batch_size = 16
lang = "fr"

# 2. Cache'lenmiş corpus embeddings yükle
collection_embeddings = torch.load('baseline/cache/collection_embeddings.pt')
collection_keys = np.load('baseline/cache/collection_keys.npy')
print(f"Collection: {collection_embeddings.shape}")

# 3. E5 retrieval modeli yükle
e5_name = 'intfloat/multilingual-e5-large'
e5_tokenizer = AutoTokenizer.from_pretrained(e5_name)
e5_model = AutoModel.from_pretrained(e5_name).to(device).eval()
print("E5 model loaded.")

# 4. Çeviri modeli yükle (FR -> EN)
mt_name = "Helsinki-NLP/opus-mt-fr-en"
mt_tokenizer = MarianTokenizer.from_pretrained(mt_name)
mt_model = MarianMTModel.from_pretrained(mt_name).to(device).eval()
print("MT model loaded.")

# 5. Yardımcı fonksiyonlar
def clean_tweet(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'^\d+/n\s*Fact\s*\d+\s*:\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

def translate_batch(texts, bs=32, max_len=256):
    out = []
    for i in tqdm(range(0, len(texts), bs), desc="Translating"):
        batch = texts[i:i+bs]
        inputs = mt_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            generated = mt_model.generate(**inputs, max_length=max_len, num_beams=1)
        decoded = mt_tokenizer.batch_decode(generated, skip_special_tokens=True)
        out.extend(decoded)
    return out

def embed_texts(texts, max_len=128):
    ds = Dataset.from_dict({"text": [f'query: {t}' for t in texts]})
    def tok(ex):
        return e5_tokenizer(ex["text"], max_length=max_len, truncation=True, padding="max_length", return_tensors="pt")
    ds = ds.map(tok, batched=True, remove_columns=ds.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
    embs = []
    with torch.inference_mode():
        for batch in tqdm(DataLoader(ds, batch_size=batch_size, num_workers=0), desc="Embedding"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = e5_model(**batch)
            embs.append(average_pool(outputs.last_hidden_state, batch['attention_mask']).detach().cpu())
    return F.normalize(torch.vstack(embs), p=2, dim=1)

# 6. Ham FR veriyi yükle
data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", lang)
train_raw, dev_raw = data["train"], data["dev"]
train_labels = np.array(train_raw["pubkey"])
dev_labels = np.array(dev_raw["pubkey"])

# 7. Temizle + çevir
train_clean = [clean_tweet(t) for t in train_raw["text"]]
dev_clean = [clean_tweet(t) for t in dev_raw["text"]]
print(f"Translating {len(train_clean)} train + {len(dev_clean)} dev tweets...")
train_en = translate_batch(train_clean)
dev_en = translate_batch(dev_clean)

# 8. Birkaç örnek
for fr, en in zip(train_clean[:3], train_en[:3]):
    print(f"FR: {fr[:120]}")
    print(f"EN: {en[:120]}")
    print("---")

# 9. Embed query
train_embeddings = embed_texts(train_en)
dev_embeddings = embed_texts(dev_en)
print(f"Train emb: {train_embeddings.shape}, Dev emb: {dev_embeddings.shape}")

# 10. MRR@5
def mrr_at_5(cos_sims, labels):
    top5_idxs = np.argsort(-cos_sims, axis=1)[:, :5]
    top5_keys = collection_keys[top5_idxs]
    labels_2d = labels[:, None]
    matches_mask = (top5_keys == labels_2d)
    positions = np.argmax(matches_mask, axis=1)
    positions[~matches_mask.any(axis=1)] = -1
    rr = np.zeros_like(positions, dtype=float)
    mask = positions >= 0
    rr[mask] = 1.0 / (positions[mask] + 1)
    return float(rr.mean())

train_cos = (train_embeddings @ collection_embeddings.T).numpy()
dev_cos = (dev_embeddings @ collection_embeddings.T).numpy()
print(f"\n=== PIVOT TRANSLATION (FR->EN) RESULTS ===")
print(f"MRR@5 Train: {mrr_at_5(train_cos, train_labels):.4f}")
print(f"MRR@5 Dev:   {mrr_at_5(dev_cos, dev_labels):.4f}")
print(f"Baseline FR Train: 0.4697 / Dev: 0.4584")

Device: mps
Collection: torch.Size([10000, 1024])


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7767.63it/s]


E5 model loaded.


/Users/ekinkeskin/air2026ss-checkthat-task1-checkthat-task1-methodology-3-group-1/.venv/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 256/256 [00:00<00:00, 21478.70it/s]


MT model loaded.
Translating 2807 train + 702 dev tweets...


Translating: 100%|██████████| 22/22 [01:38<00:00,  4.48s/it]


FR: Le SRAS-CoV-2 peut infecter immédiatement un type spécialisé de cellules rénales. La trouvaille contribue à éclaircir po
EN: SARS-CoV-2 can immediately infect a specialized type of kidney cells. Finding helps to clarify why acute renal failure i
---
FR: Une étude dernièrement diffusée dans la revue « Food and Chemical Toxicology » a aussi associé les vaccins à ARNm à une 
EN: A study recently published in the journal "Food and Chemical Toxicology" also associated mRNA vaccines with an increase 
---
FR: Dans « Prolégomènes à une histoire des collections spéciales de l’Université de Montréal », Normand Trudel et Éric Bouch
EN: In "Prolegomenes à un histoire des collections spéciales de l'Université de Montréal", Normand Trudel and Éric Bouchard 
---


Embedding: 100%|██████████| 44/44 [00:41<00:00,  1.07it/s]


Train emb: torch.Size([2807, 1024]), Dev emb: torch.Size([702, 1024])

=== PIVOT TRANSLATION (FR->EN) RESULTS ===
MRR@5 Train: 0.4611
MRR@5 Dev:   0.4618
Baseline FR Train: 0.4697 / Dev: 0.4584


In [14]:
import json, os
from datetime import datetime

results = {
    "timestamp": datetime.now().isoformat(),
    "experiment": "exp2_pivot_fr",
    "model": "intfloat/multilingual-e5-large",
    "translation_model": "Helsinki-NLP/opus-mt-fr-en",
    "language": "fr",
    "transformation": "clean + pivot translation FR->EN",
    "mrr5_train": 0.4611,
    "mrr5_dev": 0.4618,
    "delta_train": 0.4611 - 0.4697,
    "delta_dev": 0.4618 - 0.4584,
    "notes": "Translation quality mixed (some untranslated). Net effect ~zero. multilingual-e5 already handles cross-lingual well."
}

os.makedirs('baseline/results', exist_ok=True)
with open('baseline/results/results.jsonl', 'a') as f:
    f.write(json.dumps(results) + '\n')

# Embedding'leri de sakla
torch.save(train_embeddings, 'baseline/cache/train_embeddings_fr_pivot.pt')
torch.save(dev_embeddings, 'baseline/cache/dev_embeddings_fr_pivot.pt')

# Çeviri sonuçlarını da sakla — sonradan claim extraction için kullanırız
import json
with open('baseline/cache/train_en_fr.json', 'w', encoding='utf-8') as f:
    json.dump(train_en, f, ensure_ascii=False)
with open('baseline/cache/dev_en_fr.json', 'w', encoding='utf-8') as f:
    json.dump(dev_en, f, ensure_ascii=False)

print(json.dumps(results, indent=2))
print("\nSaved embeddings + translations.")

{
  "timestamp": "2026-05-03T01:18:31.961273",
  "experiment": "exp2_pivot_fr",
  "model": "intfloat/multilingual-e5-large",
  "translation_model": "Helsinki-NLP/opus-mt-fr-en",
  "language": "fr",
  "transformation": "clean + pivot translation FR->EN",
  "mrr5_train": 0.4611,
  "mrr5_dev": 0.4618,
  "delta_train": -0.008599999999999997,
  "delta_dev": 0.003400000000000014,
  "notes": "Translation quality mixed (some untranslated). Net effect ~zero. multilingual-e5 already handles cross-lingual well."
}

Saved embeddings + translations.


In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm

/Users/ekinkeskin/air2026ss-checkthat-task1-checkthat-task1-methodology-3-group-1/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = 'intfloat/multilingual-e5-large'
lang = "fr"
batch_size = 16

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: mps


In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8033.88it/s]


In [4]:
collection_embeddings = torch.load('baseline/cache/collection_embeddings.pt')
collection_keys = np.load('baseline/cache/collection_keys.npy')
print(f"Loaded cached collection embeddings: {collection_embeddings.shape}")
print(f"Collection keys: {len(collection_keys)} entries")

Loaded cached collection embeddings: torch.Size([10000, 1024])
Collection keys: 10000 entries


### Load Collection Data

In [6]:
collection_data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", "collection")["collection"]
collection_keys = np.array(collection_data["pubkey"])

def create_text(example):
    text = f'passage: {example["title"]}. {example["venue"]}. {example["abstract"]}. {example["authors"]}'
    return {"text": text}

collection_data = collection_data.map(create_text)

def tokenize_with_padding(examples):
    return tokenizer(examples["text"], max_length=512, truncation=True, padding="max_length", return_tensors="pt")

collection_data = collection_data.map(tokenize_with_padding, batched=True, remove_columns=collection_data.column_names).with_format("torch", columns=["input_ids", "attention_mask"])

### Embed Collection Data

In [7]:
def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]
    
embeddings = []
with torch.inference_mode():
    for batch in tqdm(DataLoader(collection_data, batch_size=batch_size, num_workers=0)):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
        embeddings.append(batch_embeddings.detach().cpu())

collection_embeddings = torch.vstack(embeddings)
collection_embeddings = F.normalize(collection_embeddings, p=2, dim=1)

100%|██████████| 625/625 [2:31:14<00:00, 14.52s/it]    


### Load Query Data

In [8]:
import re
from transformers import MarianMTModel, MarianTokenizer

# Çeviri modeli (FR -> EN)
mt_model_name = "Helsinki-NLP/opus-mt-fr-en"
mt_tokenizer = MarianTokenizer.from_pretrained(mt_model_name)
mt_model = MarianMTModel.from_pretrained(mt_model_name).to(device)
mt_model.eval()
print(f"Translation model loaded on {device}")

def clean_tweet(text):
    """Tweet noise removal."""
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'^\d+/n\s*Fact\s*\d+\s*:\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\d+/\d+\s*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def translate_batch(texts, batch_size=32, max_length=256):
    """FR metinlerini EN'e batch halinde çevirir."""
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Translating"):
        batch = texts[i:i+batch_size]
        inputs = mt_tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            generated = mt_model.generate(**inputs, max_length=max_length, num_beams=1)
        decoded = mt_tokenizer.batch_decode(generated, skip_special_tokens=True)
        out.extend(decoded)
    return out

ImportError: 
MarianTokenizer requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.


In [ ]:
# Önce ham tweet'leri al (cleaning + tokenize'dan ÖNCE)
data = load_dataset("sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims", lang)
train_raw = data["train"]
dev_raw = data["dev"]
train_labels = np.array(train_raw["pubkey"])
dev_labels = np.array(dev_raw["pubkey"])

# Hafif temizlik (URL/mention/hashtag), çeviri öncesi
train_texts_clean = [clean_tweet(t) for t in train_raw["text"]]
dev_texts_clean = [clean_tweet(t) for t in dev_raw["text"]]

# Çevir (uzun adım, ~3-5 dk)
print(f"Translating {len(train_texts_clean)} train tweets...")
train_texts_en = translate_batch(train_texts_clean)

print(f"Translating {len(dev_texts_clean)} dev tweets...")
dev_texts_en = translate_batch(dev_texts_clean)

# Birkaç örnek
for fr, en in list(zip(train_texts_clean[:3], train_texts_en[:3])):
    print(f"FR: {fr[:120]}")
    print(f"EN: {en[:120]}")
    print("---")

NameError: name 'train' is not defined

In [ ]:
from datasets import Dataset

# Çevrilmiş tweet'leri Dataset olarak yeniden yapılandır
train = Dataset.from_dict({"text": [f'query: {t}' for t in train_texts_en]})
dev = Dataset.from_dict({"text": [f'query: {t}' for t in dev_texts_en]})

def tokenize_with_padding(examples):
    return tokenizer(examples["text"], max_length=128, truncation=True, padding="max_length", return_tensors="pt")

train = train.map(tokenize_with_padding, batched=True, remove_columns=train.column_names).with_format("torch", columns=["input_ids", "attention_mask"])
dev = dev.map(tokenize_with_padding, batched=True, remove_columns=dev.column_names).with_format("torch", columns=["input_ids", "attention_mask"])

print(f"Train: {len(train)}, Dev: {len(dev)} (translated to EN)")

RAW:   Le SRAS-CoV-2 peut infecter immédiatement un type spécialisé de cellules rénales. La trouvaille contribue à éclaircir pourquoi l'i
CLEAN: Le SRAS-CoV-2 peut infecter immédiatement un type spécialisé de cellules rénales. La trouvaille contribue à éclaircir pourquoi l'i
---
RAW:   Une étude dernièrement diffusée dans la revue « Food and Chemical Toxicology » a aussi associé les vaccins à ARNm à une hausse des
CLEAN: Une étude dernièrement diffusée dans la revue Food and Chemical Toxicology a aussi associé les vaccins à ARNm à une hausse des can
---
RAW:   Dans « Prolégomènes à une histoire des collections spéciales de l’Université de Montréal », Normand Trudel et Éric Bouchard suggèr
CLEAN: Dans Prolégomènes à une histoire des collections spéciales de l Université de Montréal , Normand Trudel et Éric Bouchard suggèrent
---
RAW:   Une récente étude démontre que le vieillissement épigénétique rapide est lié au danger de contamination au SARS‑CoV‑2 et aussi de 
CLEAN: Une récente étu

In [17]:
from torch import Tensor

def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

### Embed Query Data

In [18]:
train_embeddings = []
with torch.inference_mode():
    for batch in tqdm(DataLoader(train, batch_size=batch_size, num_workers=0)):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
        train_embeddings.append(batch_embeddings.detach().cpu())

train_embeddings = torch.vstack(train_embeddings)
train_embeddings = F.normalize(train_embeddings, p=2, dim=1)

dev_embeddings = []
with torch.inference_mode():
  for batch in tqdm(DataLoader(dev, batch_size=batch_size, num_workers=0)):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        batch_embeddings = average_pool(outputs.last_hidden_state, batch['attention_mask'])
        dev_embeddings.append(batch_embeddings.detach().cpu())

dev_embeddings = torch.vstack(dev_embeddings)
dev_embeddings = F.normalize(dev_embeddings, p=2, dim=1)

100%|██████████| 44/44 [00:39<00:00,  1.12it/s]


### Calculate MRR@5 Score

In [8]:
def mrr_at_5(cos_sims, labels):
    # Get top-5 predictions
    top5_idxs = np.argsort(-cos_sims, axis=1)[:, :5]
    top5_keys = collection_keys[top5_idxs]
    
    # Compare labels to top-5 predictions
    labels_2d = labels[:, None]
    matches_mask = (top5_keys == labels_2d)

    # Get position of correct predictions
    positions = np.argmax(matches_mask, axis=1)
    positions[~matches_mask.any(axis=1)] = -1
    positions = positions.astype(int)

    # Get reciprocal rank
    rr = np.zeros_like(positions, dtype=float)
    mask = positions >= 0
    rr[mask] = 1.0 / (positions[mask] + 1)

    return float(rr.mean())

In [11]:
print("train_embeddings var mı?", 'train_embeddings' in dir())
print("dev_embeddings var mı?", 'dev_embeddings' in dir())
print("collection_embeddings var mı?", 'collection_embeddings' in dir())
print("train var mı?", 'train' in dir())
print("dev var mı?", 'dev' in dir())

train_embeddings var mı? False
dev_embeddings var mı? False
collection_embeddings var mı? True
train var mı? False
dev var mı? False


In [12]:
print("train_texts_en var mı?", 'train_texts_en' in dir())
print("dev_texts_en var mı?", 'dev_texts_en' in dir())
print("mt_model var mı?", 'mt_model' in dir())

train_texts_en var mı? False
dev_texts_en var mı? False
mt_model var mı? False


In [10]:
train_cos_sims = (train_embeddings @ collection_embeddings.T).numpy()
dev_cos_sims = (dev_embeddings @ collection_embeddings.T).numpy()

print("MRR@5 Train:", mrr_at_5(train_cos_sims, train_labels))
print("MRR@5 Dev:", mrr_at_5(dev_cos_sims, dev_labels))

NameError: name 'train_embeddings' is not defined

In [21]:
import json, os
from datetime import datetime

os.makedirs('baseline/results', exist_ok=True)

new_train = 0.4778945493409334
new_dev = 0.461965811965812

results = {
    "timestamp": datetime.now().isoformat(),
    "experiment": "exp1_clean_fr",
    "model": "intfloat/multilingual-e5-large",
    "language": "fr",
    "transformation": "rule_based_cleaning",
    "mrr5_train": new_train,
    "mrr5_dev": new_dev,
    "delta_train": new_train - 0.4697,
    "delta_dev": new_dev - 0.4584,
    "notes": "URL/@/# remove + emoji strip + numbering pattern removal. Note: SARS-CoV-2 splits incorrectly (Unicode dash issue, to fix)."
}

with open('baseline/results/results.jsonl', 'a') as f:
    f.write(json.dumps(results) + '\n')

print(json.dumps(results, indent=2))

{
  "timestamp": "2026-05-02T15:42:48.021936",
  "experiment": "exp1_clean_fr",
  "model": "intfloat/multilingual-e5-large",
  "language": "fr",
  "transformation": "rule_based_cleaning",
  "mrr5_train": 0.4778945493409334,
  "mrr5_dev": 0.461965811965812,
  "delta_train": 0.008194549340933388,
  "delta_dev": 0.0035658119658120158,
  "notes": "URL/@/# remove + emoji strip + numbering pattern removal. Note: SARS-CoV-2 splits incorrectly (Unicode dash issue, to fix)."
}


In [22]:
torch.save(train_embeddings, 'baseline/cache/train_embeddings_fr_clean.pt')
torch.save(dev_embeddings, 'baseline/cache/dev_embeddings_fr_clean.pt')
print("Saved cleaned FR query embeddings.")

Saved cleaned FR query embeddings.


In [23]:
import json, os
from datetime import datetime

os.makedirs('baseline/results', exist_ok=True)

new_train = 0.4778945493409334
new_dev = 0.461965811965812

results = {
    "timestamp": datetime.now().isoformat(),
    "experiment": "exp1_clean_fr",
    "model": "intfloat/multilingual-e5-large",
    "language": "fr",
    "transformation": "rule_based_cleaning",
    "mrr5_train": new_train,
    "mrr5_dev": new_dev,
    "delta_train": new_train - 0.4697,
    "delta_dev": new_dev - 0.4584,
    "notes": "URL/@/# remove + emoji strip + numbering pattern removal."
}

with open('baseline/results/results.jsonl', 'a') as f:
    f.write(json.dumps(results) + '\n')

print(json.dumps(results, indent=2))

{
  "timestamp": "2026-05-02T15:43:33.478621",
  "experiment": "exp1_clean_fr",
  "model": "intfloat/multilingual-e5-large",
  "language": "fr",
  "transformation": "rule_based_cleaning",
  "mrr5_train": 0.4778945493409334,
  "mrr5_dev": 0.461965811965812,
  "delta_train": 0.008194549340933388,
  "delta_dev": 0.0035658119658120158,
  "notes": "URL/@/# remove + emoji strip + numbering pattern removal."
}
